# Kaggle — YOLOv8n × 3 seeds + 2 cross-country runs (2× T4, in parallel)
Replaces the Colab notebook. **Settings:** Accelerator **GPU T4 ×2**, Internet **On**.
Run with **Save Version → Save & Run All (Commit)**. Takes about 3 hours.
At the end download **`results_colab.zip`** from Output (named that way so `laptop.sh finish` picks it up).

In [ ]:
import os, subprocess, shutil
REPO_URL = "https://github.com/AdonisYsh/road-defect-severity.git"   # <- your GitHub repo (must be public, or put a token in the URL)
ROOT = "/kaggle/working/road-defect"
if os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "-C", ROOT, "pull", "--ff-only"], check=True)
else:
    tmp = ROOT + "_clone"
    shutil.rmtree(tmp, ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, tmp], check=True)
    os.makedirs(ROOT, exist_ok=True)
    shutil.copytree(tmp, ROOT, dirs_exist_ok=True)   # keeps anything the preflight left (data/raw, weights/)
    shutil.rmtree(tmp)
os.chdir(ROOT)
subprocess.run("pip install -q -r requirements-cloud.txt", shell=True, check=True)
print("repo ready at", os.getcwd())

In [ ]:
!python -m rdd.download && python -m rdd.prep

In [ ]:
import os, subprocess, time, sys, pathlib, yaml
# ---- time budget per job (hours). Total wall time ~ max(queue sums) + ~15 min eval per job.
SEED_HOURS, XC_HOURS = 1.0, 0.8
cfg_p = pathlib.Path("config.yaml"); c = yaml.safe_load(cfg_p.read_text())
for j in ("yolov8n_seed0", "yolov8n_seed1", "yolov8n_seed2"):
    c["jobs"][j]["time_hours"] = SEED_HOURS
for j in ("xc_india", "xc_japan"):
    c["jobs"][j]["time_hours"] = XC_HOURS
c["yolo"]["workers"] = 2           # 4 CPUs shared by two trainings
cfg_p.write_text(yaml.safe_dump(c, sort_keys=False))
QUEUES = {"0": ["yolov8n_seed0", "yolov8n_seed1"],               # GPU 0: ~2.0 h
          "1": ["yolov8n_seed2", "xc_india", "xc_japan"]}         # GPU 1: ~2.6 h
procs = {}
for k, (gpu, jobs) in enumerate(QUEUES.items()):
    if k:
        time.sleep(180)   # stagger so the two runs don't build the label cache at the same moment
    log = open(f"log_gpu{gpu}.txt", "w")
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpu)   # each queue sees only its own GPU (as cuda:0)
    procs[gpu] = subprocess.Popen([sys.executable, "-m", "rdd.jobs", *jobs], env=env,
                                  stdout=log, stderr=subprocess.STDOUT)
    print(f"GPU {gpu}: started {jobs}  (log: log_gpu{gpu}.txt)", flush=True)
while any(p.poll() is None for p in procs.values()):
    time.sleep(600)
    for gpu in procs:
        lines = [l for l in pathlib.Path(f"log_gpu{gpu}.txt").read_text(errors="ignore").splitlines() if l.strip()]
        print(f"[GPU {gpu}] {lines[-1][-160:] if lines else '...'}", flush=True)
for gpu, p in procs.items():
    print(f"GPU {gpu} finished with exit code {p.returncode}")
    print("".join(open(f"log_gpu{gpu}.txt", errors="ignore").readlines()[-15:]))
assert all(p.returncode == 0 for p in procs.values()), "a queue failed - see the log above"

In [ ]:
!python -m rdd.benchmark

In [ ]:
!python -m rdd.report --pack
!mv results_kaggle.zip results_colab.zip
!rm -rf data runs/preds
!ls -lh results_colab.zip